In [ ]:
# Configuração do ambiente

# Garante que o notebook está na raiz do projeto
%cd .. 

# Verifica o diretório atual (Linux/Mac)
# !pwd  

# Verifica o diretório atual (Windows)
!cd 

In [ ]:
# Importações necessárias 

import numpy as np
import pandas as pd
from termcolor import colored

from src.serving.inference import load_inference_bundle, predict_dataframe

In [ ]:
# Carregar o bundle de inferência de produção (Pipeline + threshold)
# O Pipeline aplica internamente a transformação de Amount (PowerTransformer)
# e o MinMaxScaler de Time, exatamente como no treino.
bundle = load_inference_bundle()

print(f"Threshold operacional: {bundle.threshold:.4f}")
print(f"Features esperadas: {bundle.feature_names}")

In [ ]:
# Gerar e testar 40 transações aleatórias (determinístico: SEED fixa)
SEED = 42
rng = np.random.default_rng(SEED)

for i in range(40):
    nova_transacao = pd.DataFrame({
        'Time': [rng.uniform(0, 172792)],  # Segundos desde a 1ª transação (escala bruta)
        'V1': [rng.uniform(-5, 5)],
        'V2': [rng.uniform(-5, 5)],
        'V3': [rng.uniform(-5, 5)],
        'V4': [rng.uniform(-5, 5)],
        'V5': [rng.uniform(-5, 5)],
        'V6': [rng.uniform(-5, 5)],
        'V7': [rng.uniform(-5, 5)],
        'V8': [rng.uniform(-5, 5)],
        'V9': [rng.uniform(-5, 5)],
        'V10': [rng.uniform(-5, 5)],
        'V11': [rng.uniform(-5, 5)],
        'V12': [rng.uniform(-5, 5)],
        'V13': [rng.uniform(-5, 5)],
        'V14': [rng.uniform(-5, 5)],
        'V15': [rng.uniform(-5, 5)],
        'V16': [rng.uniform(-5, 5)],
        'V17': [rng.uniform(-5, 5)],
        'V18': [rng.uniform(-5, 5)],
        'V19': [rng.uniform(-5, 5)],
        'V20': [rng.uniform(-5, 5)],
        'V21': [rng.uniform(-5, 5)],
        'V22': [rng.uniform(-5, 5)],
        'V23': [rng.uniform(-5, 5)],
        'V24': [rng.uniform(-5, 5)],
        'V25': [rng.uniform(-5, 5)],
        'V26': [rng.uniform(-5, 5)],
        'V27': [rng.uniform(-5, 5)],
        'V28': [rng.uniform(-5, 5)],
        'Amount': [rng.uniform(1, 5000)]  # Valores entre 1 e 5000 reais
    })
    
    # Inferência pelo contrato de produção (o Pipeline aplica o preprocessamento do treino)
    previsao = predict_dataframe(nova_transacao, bundle)
    probabilidade = previsao['fraud_probability'][0]
    fraudou = previsao['fraud_prediction'][0] == 1
    
    # Exibir resultado com destaque em vermelho para fraudes
    resultado = (f"Teste {i+1}: Transação de R${nova_transacao['Amount'][0]:.2f} "
                 f"(p={probabilidade:.4f}) - "
                 "Fraude detectada!" if fraudou else "Transação segura.")
    
    if fraudou:
        print(colored(resultado, 'red'))
    else:
        print(resultado)